# Plant Disease VQA — Google Colab training

Run cells manually in order. No benchmark launches automatically. Project code and temporary images live under /content; only durable artifacts are copied to Drive. Runtime setup and training cells require a Colab GPU.

## 1. Choose the project source

Set a public GitHub URL or upload a project ZIP/folder through the Files panel. Pin PROJECT_REF to a commit for reproducible runs. Do not clone into Drive.

In [26]:
from pathlib import Path
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"  # PyTorch-only Transformers backend

PROJECT_URL = "https://github.com/dinhdat081026/PlantDisease_VQA.git"  # replace
PROJECT_REF = "main"  # preferably pin a commit SHA
UPLOAD_ZIP = ""  # optional: /content/plant-vqa-project.zip
PROJECT_DIR = Path("/content/plant_disease_visual_question_answering")
DATA_DIR = Path("/content/data/PlantVillageVQA")
OUTPUT_DIR = Path("/content/outputs")

if UPLOAD_ZIP:
    import zipfile
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(UPLOAD_ZIP) as zf:
        root = PROJECT_DIR.resolve()
        for member in zf.infolist():
            target = (PROJECT_DIR / member.filename).resolve()
            if not target.is_relative_to(root) or ".." in Path(member.filename).parts:
                raise ValueError(f"Unsafe project archive member: {member.filename}")
        zf.extractall(PROJECT_DIR)
elif not (PROJECT_DIR / "src/train.py").exists():
    if "OWNER" in PROJECT_URL:
        raise ValueError("Set PROJECT_URL to your GitHub repository or upload the project ZIP")
    !git clone --branch {PROJECT_REF} --depth 1 {PROJECT_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)
print("Data:", DATA_DIR)
print("Outputs:", OUTPUT_DIR)

Project: /content/plant_disease_visual_question_answering
Data: /content/data/PlantVillageVQA
Outputs: /content/outputs


## 2. Verify accelerator and install dependencies

This project is a source tree with a `src/` package, but no pyproject.toml/setup.py and no installed console entry points. Editable installation is unnecessary: run `python -m src.train` from PROJECT_DIR and add PROJECT_DIR to sys.path for notebook imports. The pip command deliberately shows full diagnostics. It never reinstalls torch or torchvision.

In [27]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
import subprocess, sys, importlib.util

for module in ("torch", "torchvision"):
    if importlib.util.find_spec(module) is None:
        raise RuntimeError(f"Colab runtime is missing {module}; select a CUDA-enabled runtime")
import torch
print("Python:", sys.version)
# Python 3.13 is acceptable only when Colab’s preinstalled torch/torchvision
# are a mutually compatible CUDA build. The checked-in 2.6.0/0.21.0 pins
# have CPython 3.13 wheels, but this cell intentionally does not install them.
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime (Runtime > Change runtime type > T4 GPU)")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

# Keep the active CUDA PyTorch/torchvision pair untouched.
requirements = (PROJECT_DIR / "requirements.txt").read_text().splitlines()
install = [line.strip() for line in requirements if line.strip() and not line.lstrip().startswith("#") and line.split("==")[0].strip().lower() not in {"torch", "torchvision"}]
subprocess.check_call([sys.executable, "-m", "pip", "install", *install])

# Source-tree import: no editable install and no packaging metadata required.
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
import src.config, src.data, src.models, src.train, src.evaluate
print("Project modules imported from:", PROJECT_DIR)
print("No editable install is required.")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130
CUDA runtime: 13.0
CUDA available: True
GPU: Tesla T4
VRAM GiB: 14.56
Project modules imported from: /content/plant_disease_visual_question_answering
No editable install is required.


## 3. Mount Drive for durable storage

In [28]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/PlantDiseaseVQA")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Durable outputs:", DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Durable outputs: /content/drive/MyDrive/PlantDiseaseVQA


## 4. Download or locate dataset

Dataset is stored temporarily under /content. Download uses the repository CLI and Hugging Face revision discovery. To use an uploaded archive, extract it into DATA_DIR and verify it has one CSV with the six source columns. Never put the dataset in Drive for training.

In [29]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
source_csv = sorted(DATA_DIR.rglob("*.csv"))
if not source_csv:
    subprocess.check_call([sys.executable, "-m", "src.download", "--data_dir", str(DATA_DIR)], cwd=PROJECT_DIR)
source_csv = list(DATA_DIR.rglob("*.csv"))
if len(source_csv) != 1:
    raise RuntimeError(f"Expected exactly one discovered annotation CSV, got {source_csv}; set an explicit --csv in later cells")
SOURCE_CSV = source_csv[0]
print("Annotation:", SOURCE_CSV)
print("Image files:", sum(1 for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}))

Annotation: /content/data/PlantVillageVQA/a025a438308eff9a7289231ebc09e27e8ec6678c/PlantVillageVQA/PlantVillageVQA.csv
Image files: 55448


## 5. EDA, tests and a model smoke test

Run EDA and tests before training. Then construct a fresh-initialized batch-1 ViT+BERT model and run one inference forward to check actual imports, tensor wiring and CUDA memory. This does not train or save a checkpoint.

In [30]:
subprocess.check_call([sys.executable, "-m", "src.eda", "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV), "--output_dir", str(OUTPUT_DIR)], cwd=PROJECT_DIR)
subprocess.check_call([sys.executable, "-m", "src.verify_numbers", "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV), "--output", str(OUTPUT_DIR / "results/verification.json")], cwd=PROJECT_DIR)

# Make real-data tests inspect the CSV downloaded in this notebook instead of
# silently skipping because it is outside the repository's default data path.
test_env = os.environ.copy()
test_env["PLANTVQA_CSV"] = str(SOURCE_CSV)
test_command = [sys.executable, "-m", "pytest", "tests/test_data.py", "tests/test_dataset_pipeline.py", "tests/test_models.py", "tests/test_train_evaluate.py", "-v"]
print("Running:", " ".join(test_command))
test_result = subprocess.run(test_command, cwd=PROJECT_DIR, env=test_env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(test_result.stdout, end="" if test_result.stdout.endswith("\n") else "\n")
if test_result.returncode:
    raise RuntimeError(f"pytest failed with exit code {test_result.returncode}; see the complete pytest failure output above.")

# Tiny random-weight forward smoke test; no optimizer, gradients or training.
import torch
from src.models import ViTBERT
smoke_model = ViTBERT(num_classes=2, mode="full", pretrained=False).to("cuda").eval()
smoke_batch = {
    "pixel_values": torch.zeros(1, 3, 224, 224, device="cuda"),
    "input_ids": torch.tensor([[101, 102]], dtype=torch.long, device="cuda"),
    "attention_mask": torch.ones(1, 2, dtype=torch.long, device="cuda"),
    "token_type_ids": torch.zeros(1, 2, dtype=torch.long, device="cuda"),
}
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    smoke_logits = smoke_model(**smoke_batch)
assert smoke_logits.shape == (1, 2) and torch.isfinite(smoke_logits).all()
print("CUDA ViTBERT smoke test OK:", smoke_logits.shape, smoke_logits.dtype)
del smoke_model, smoke_batch, smoke_logits
torch.cuda.empty_cache()

Running: /usr/bin/python3 -m pytest tests/test_data.py tests/test_dataset_pipeline.py tests/test_models.py tests/test_train_evaluate.py -v
============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.3.5, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/plant_disease_visual_question_answering
plugins: langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0, platformdirs-4.14.3
collecting ... collected 83 items

tests/test_data.py::test_filter_exact_four_categories_and_no_mutation PASSED [  1%]
tests/test_data.py::test_split_disjoint_complete_test_preserved_validation_from_train PASSED [  2%]
tests/test_data.py::test_deterministic_seed_and_input_order_independence PASSED [  3%]
tests/test_data.py::test_stratified_selection_and_rare_species PASSED    [  4%]
tests/test_data.py::test_species_evidence_never_infers_from_negative_question PASSED [  6%]
tests/test_data.py::test_default_subset_preserves_every_test

## 6. Set the Colab resource profile

The project base batch size is 16 with accumulation 1. This notebook uses microbatch 4 with accumulation 4 (effective batch size 16) to reduce T4 activation memory. This is an explicit runtime override; it changes microbatch execution and dropout sampling. If VRAM is still insufficient, use batch 2/accumulation 8, then batch 1/accumulation 16.

In [31]:
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
NUM_WORKERS = 2

# Explicit override only: keep the original official test whenever possible.
TEST_POLICY = "subset"  # mini/class total budgets are smaller than all factoid test images

assert BATCH_SIZE * GRADIENT_ACCUMULATION == 16
print({"microbatch": BATCH_SIZE, "gradient_accumulation_steps": GRADIENT_ACCUMULATION,
       "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION, "test_policy": TEST_POLICY})

{'microbatch': 4, 'gradient_accumulation_steps': 4, 'effective_batch_size': 16, 'test_policy': 'subset'}


## 7. Validate data for each planned run

Run this cell before each configuration. Validation runs selected-image decoding and pytest without skipped tests; no model is initialized in validation-only mode. It can take several minutes for full data.

In [32]:
def validate_run(level, mode):
    report = OUTPUT_DIR / "results" / f"data_validation_{level}_{TEST_POLICY}.json"
    cmd = [sys.executable, "-u", "-m", "src.train", "--level", level, "--mode", mode,
           "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV),
           "--output-dir", str(OUTPUT_DIR), "--test-policy", TEST_POLICY,
           "--batch-size", str(BATCH_SIZE), "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION),
           "--num-workers", str(NUM_WORKERS), "--validate-only", "--validation-report", str(report)]
    print("Validating:", " ".join(cmd), flush=True)
    # Capture merged output so Colab displays argparse errors and data failures.
    completed = subprocess.run(cmd, cwd=PROJECT_DIR, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(completed.stdout, flush=True)
    if completed.returncode:
        raise RuntimeError(
            f"Data validation failed (exit {completed.returncode}). "
            "See the complete output above; training has not started. "
            f"If pytest ran, also inspect {report.with_suffix('.pytest.log')}")
    return report

print("Validation helper ready. No run has started.")


Validation helper ready. No run has started.


## 8. Train the selected configuration

Run the helper cell, Step 1 Training (default mini/full), then Step 2 checkpoint verification and Drive backup. Running Step 1 actually starts GPU training. Select another level/mode there for subsequent experiments. Existing names use mode_level, for example full_mini.pt; _last.pt is for resume only.


In [33]:
def best_checkpoint(level, mode):
    # Match src.train's existing mode_level naming, preserving resume compatibility.
    return OUTPUT_DIR / "models" / f"{mode}_{level}.pt"

def check_best_checkpoint(path, level, mode):
    from src.evaluate import load_checkpoint
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Best checkpoint missing or empty: {path}. Run the Training cell first.")
    state = load_checkpoint(path)
    if state["config"]["level"] != level or state["mode"] != mode:
        raise ValueError("Checkpoint configuration differs from selected LEVEL/MODE")
    print("Best checkpoint:", path, "bytes:", path.stat().st_size,
          "epoch:", state["epoch"], "validation accuracy:", state["validation_metric"]["value"])
    return path

def backup_outputs():
    import shutil
    for folder in ("models", "results", "figures"):
        source = OUTPUT_DIR / folder
        if source.exists():
            destination = DRIVE_ROOT / folder
            destination.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, destination, dirs_exist_ok=True)
    print("Backup destination:", DRIVE_ROOT)

def train_run(level, mode="full", *, resume=False):
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU runtime before training")
    report = validate_run(level, mode)
    cmd = [sys.executable, "-u", "-m", "src.train", "--level", level, "--mode", mode,
           "--device", "cuda", "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV),
           "--output-dir", str(OUTPUT_DIR), "--test-policy", TEST_POLICY,
           "--batch-size", str(BATCH_SIZE), "--gradient-accumulation-steps", str(GRADIENT_ACCUMULATION),
           "--num-workers", str(NUM_WORKERS), "--validation-report", str(report)]
    if resume:
        cmd.append("--resume")
    print("Training:", " ".join(cmd), flush=True)
    try:
        subprocess.check_call(cmd, cwd=PROJECT_DIR)
    finally:
        # Preserve completed epoch checkpoints even if a later epoch fails.
        backup_outputs()
    return check_best_checkpoint(best_checkpoint(level, mode), level, mode)

print("Training helper ready. Run the next cell to train the selected configuration.")


Training helper ready. Run the next cell to train the selected configuration.


In [34]:
# Step 1 — Training. Running this cell launches one selected experiment.
LEVEL = "mini"  # mini, class, full
MODE = "full"   # full, frozen, question_only, image_only
RESUME = False  # True only for the matching local _last.pt checkpoint
TRAINED_CHECKPOINT = train_run(LEVEL, MODE, resume=RESUME)


Validating: /usr/bin/python3 -u -m src.train --level mini --mode full --data_dir /content/data/PlantVillageVQA --csv /content/data/PlantVillageVQA/a025a438308eff9a7289231ebc09e27e8ec6678c/PlantVillageVQA/PlantVillageVQA.csv --output-dir /content/outputs --test-policy subset --batch-size 4 --gradient-accumulation-steps 4 --num-workers 2 --validate-only --validation-report /content/outputs/results/data_validation_mini_subset.json
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
{
  "passed": true,
  "seed": 59,
  "data_manifest": {
    "source_sha256": "a96f8b5306bac41e171ad36af7ed3b2c0d0fd4ca4d3b2eacc06617ef8fafc06d",
    "image_fingerprint": "687e1a82b0e24613e1774165de9069fb07ed656206067bf70a3ff2dfee6b449a",
    "image_count": 993,
    "data_code_sha256": {
      "src/config.py": "0c7bca542ee1a34cd9f5fd24442c7fe068fb130591c5add3f82bef1a5ec691fd",
      "src/data.py

In [35]:
# Step 2 — Verify and save the best checkpoint to Drive.
# Training already writes best and last checkpoints under OUTPUT_DIR/models.
CHECKPOINT = check_best_checkpoint(best_checkpoint(LEVEL, MODE), LEVEL, MODE)
backup_outputs()
DRIVE_CHECKPOINT = DRIVE_ROOT / "models" / CHECKPOINT.name
if not DRIVE_CHECKPOINT.is_file() or DRIVE_CHECKPOINT.stat().st_size != CHECKPOINT.stat().st_size:
    raise RuntimeError(f"Drive checkpoint backup missing or incomplete: {DRIVE_CHECKPOINT}")
print("Saved to Drive:", DRIVE_CHECKPOINT)


Best checkpoint: /content/outputs/models/full_mini.pt bytes: 785111585 epoch: 3 validation accuracy: 0.8650306748466258
Backup destination: /content/drive/MyDrive/PlantDiseaseVQA
Saved to Drive: /content/drive/MyDrive/PlantDiseaseVQA/models/full_mini.pt


In [36]:
# To run another experiment, change LEVEL/MODE in Step 1 and execute it explicitly.


In [37]:
# To run another experiment, change LEVEL/MODE in Step 1 and execute it explicitly.


In [38]:
# To run another experiment, change LEVEL/MODE in Step 1 and execute it explicitly.


In [39]:
# To run another experiment, change LEVEL/MODE in Step 1 and execute it explicitly.


## 9. Evaluate the selected configuration

Run after Training and checkpoint verification. Selection uses LEVEL/MODE or an explicit path, never modification time. Dataset and preprocessing compatibility are checked by the evaluation CLI.


In [40]:
# Step 3 — Evaluation. This cell never launches training.
# Set an explicit path to select another best checkpoint (never an automatic _last).
CHECKPOINT_OVERRIDE = None  # e.g. DRIVE_ROOT / "models/full_mini.pt"
selected_level = globals().get("LEVEL", "mini")
selected_mode = globals().get("MODE", "full")
candidates = [best_checkpoint(selected_level, selected_mode),
              DRIVE_ROOT / "models" / f"{selected_mode}_{selected_level}.pt"]
checkpoint = Path(CHECKPOINT_OVERRIDE) if CHECKPOINT_OVERRIDE else next(
    (p for p in candidates if p.is_file() and p.stat().st_size > 0), None)
if checkpoint is None:
    print("No best checkpoint for", selected_level, selected_mode)
    print("Run Step 1 — Training, then Step 2 — Save Best Checkpoint before Evaluation.")
    print("Searched:", *candidates, sep="\n")
else:
    checkpoint = check_best_checkpoint(checkpoint, selected_level, selected_mode)
    # CLI checks annotation hashes, image content, split, vocab and preprocessing.
    subprocess.check_call([sys.executable, "-u", "-m", "src.evaluate", "--ckpt", str(checkpoint),
        "--data_dir", str(DATA_DIR), "--csv", str(SOURCE_CSV),
        "--output_dir", str(OUTPUT_DIR / "results")], cwd=PROJECT_DIR)
    backup_outputs()


Best checkpoint: /content/outputs/models/full_mini.pt bytes: 785111585 epoch: 3 validation accuracy: 0.8650306748466258
Backup destination: /content/drive/MyDrive/PlantDiseaseVQA


## 10. Copy durable results to Drive

In [41]:
backup_outputs()
print("Drive backup complete:", DRIVE_ROOT)
print("Checkpoints:", sorted(p.name for p in (DRIVE_ROOT / "models").glob("*.pt")))
print("Results:", len(list((DRIVE_ROOT / "results").glob("*"))))

Backup destination: /content/drive/MyDrive/PlantDiseaseVQA
Drive backup complete: /content/drive/MyDrive/PlantDiseaseVQA
Checkpoints: ['full_mini.pt', 'full_mini_last.pt']
Results: 13
